# Pipeline de Balanceo de Datos para Detección de Vishing

Este notebook implementa un pipeline robusto para contrarrestar el desbalance de clases del dataset (5% de fraude original), generando nuevos datasets usando diversas técnicas:
- **Random Oversampling**
- **SMOTE** (Synthetic Minority Over-sampling Technique)
- **Borderline SMOTE**
- **SMOTE + Undersampling** (Híbrido)
- **CTGAN** (Conditional Tabular GAN - basado en Deep Learning)

Generaremos versiones con proporciones de **10%**, **20%** y **25%** de la clase minoritaria (`is_vishing`).

## 1. Importar Librerías y Configurar Entorno

In [2]:
import os
import pandas as pd
import numpy as np

# Imblearn - Oversampling e Híbridos
from imblearn.over_sampling import RandomOverSampler, SMOTE, BorderlineSMOTE
from imblearn.under_sampling import RandomUnderSampler
from imblearn.pipeline import Pipeline

# SDV - GAN Tabular sintética
from sdv.single_table import CTGANSynthesizer
from sdv.metadata import SingleTableMetadata

import warnings
warnings.filterwarnings('ignore')

print('Librerías cargadas correctamente.')

Librerías cargadas correctamente.


## 2. Cargar y Preparar Dataset Original

In [3]:
RAW_DATA_PATH = 'raw_data/dataset_sintetico_biocatch_vishing_overlap.csv'
df_original = pd.read_csv(RAW_DATA_PATH)

# Excluiremos features que el EDA recomendó descartar o que no son interpolables (timestamps brutos)
# Mantenemos las numéricas para aplicar SMOTE directo
exclude_cols = ['session_id', 'customer_id', 'session_timestamp', 'device_type', 'claim_category', 
                'biocatch_risk_score', 'biocatch_genuine_score', 'biocatch_ato_indicator', 
                'biocatch_social_eng_indicator', 'biocatch_bot_indicator','device_type','os_type', 'app_version']

target_col = 'is_vishing'

features = [c for c in df_original.columns if c not in exclude_cols and c != target_col]

X = df_original[features].fillna(0) # Simple fill n/a for processing
y = df_original[target_col]

majority_count = sum(y == 0)
minority_count = sum(y == 1)
total_count = len(y)

print(f"Total de registros: {total_count}")
print(f"Clase Mayoritaria (Legítimo, 0): {majority_count} ({majority_count/total_count:.2%})")
print(f"Clase Minoritaria (Vishing,  1): {minority_count} ({minority_count/total_count:.2%})")

Total de registros: 50000
Clase Mayoritaria (Legítimo, 0): 47500 (95.00%)
Clase Minoritaria (Vishing,  1): 2500 (5.00%)


## 3. Definir Funciones Generales del Pipeline

In [4]:
def get_sampling_strategy(pct, majority_samples):
    """
    Calcula la proporción para imblearn (minority / majority) dado un porcentaje objetivo 
    para la clase minoritaria en el conjunto completo.
    Ecuación: N_min = (pct / (1 - pct)) * N_maj
    """
    ratio = pct / (1.0 - pct)
    return ratio

def save_dataset(X_res, y_res, technique, pct, base_dir='data'):
    """
    Recibe matriz X e y remuestreadas, las concatena y exporta al sistema de archivos.
    Formato esperado: data/tecnica/porcentaje.csv
    """
    dir_path = os.path.join(base_dir, technique)
    os.makedirs(dir_path, exist_ok=True)
    
    # Formatear el nombre del archivo (ej. 10.csv)
    pct_str = f"{int(pct * 100)}"
    file_path = os.path.join(dir_path, f"{pct_str}.csv")
    
    df_out = pd.concat([X_res.copy(), y_res.copy()], axis=1)
    df_out.to_csv(file_path, index=False)
    print(f"✅ Guardado: {file_path} (Distribución Vishing: {df_out['is_vishing'].mean():.2%}) | Total: {len(df_out)} filas")

TARGET_PERCENTAGES = [0.10, 0.20, 0.25]

## 4. Random Oversampling

In [5]:
print("Generando Oversampling Aleatorio...")
for pct in TARGET_PERCENTAGES:
    strategy = get_sampling_strategy(pct, majority_count)
    ros = RandomOverSampler(sampling_strategy=strategy, random_state=42)
    X_res, y_res = ros.fit_resample(X, y)
    save_dataset(X_res, y_res, 'random_oversampling', pct)

Generando Oversampling Aleatorio...
✅ Guardado: data\random_oversampling\10.csv (Distribución Vishing: 10.00%) | Total: 52777 filas
✅ Guardado: data\random_oversampling\20.csv (Distribución Vishing: 20.00%) | Total: 59375 filas
✅ Guardado: data\random_oversampling\25.csv (Distribución Vishing: 25.00%) | Total: 63333 filas


## 5. SMOTE

In [6]:
print("Generando SMOTE...")
for pct in TARGET_PERCENTAGES:
    strategy = get_sampling_strategy(pct, majority_count)
    smote = SMOTE(sampling_strategy=strategy, random_state=42)
    X_res, y_res = smote.fit_resample(X, y)
    save_dataset(X_res, y_res, 'smote', pct)

Generando SMOTE...
✅ Guardado: data\smote\10.csv (Distribución Vishing: 10.00%) | Total: 52777 filas
✅ Guardado: data\smote\20.csv (Distribución Vishing: 20.00%) | Total: 59375 filas
✅ Guardado: data\smote\25.csv (Distribución Vishing: 25.00%) | Total: 63333 filas


## 6. Borderline SMOTE

In [7]:
print("Generando Borderline SMOTE...")
for pct in TARGET_PERCENTAGES:
    strategy = get_sampling_strategy(pct, majority_count)
    bsmote = BorderlineSMOTE(sampling_strategy=strategy, random_state=42, kind='borderline-1')
    X_res, y_res = bsmote.fit_resample(X, y)
    save_dataset(X_res, y_res, 'borderline_smote', pct)

Generando Borderline SMOTE...
✅ Guardado: data\borderline_smote\10.csv (Distribución Vishing: 10.00%) | Total: 52777 filas
✅ Guardado: data\borderline_smote\20.csv (Distribución Vishing: 20.00%) | Total: 59375 filas
✅ Guardado: data\borderline_smote\25.csv (Distribución Vishing: 25.00%) | Total: 63333 filas


## 7. SMOTE + Undersampling (Pipeline)

In [8]:
print("Generando SMOTE + Undersampling...")
# Estrategia: subimos Vishing drásticamente, y bajamos la clase mayoritaria.
for pct in TARGET_PERCENTAGES:
    # Como usamos undersampling también, N_maj va a disminuir. 
    # Fijemos una reducción de la clase mayoritaria del 10% para acelerar entrenamientos (opcional),
    # y luego calculamos la proporción de la minoritaria.
    # target_majority = int(majority_count * 0.90)
    # target_minority = int(target_majority * (pct / (1 - pct)))
    
    # O simplemente dictamos el dicc de estrategias exactas (N_samples)
    target_majority_count = int(majority_count * 0.9)  # Bajamos un 10% legitimas
    target_minority_count = int(target_majority_count * (pct / (1 - pct)))
    
    # Pipeline con SMOTE (oversample minority a target_minority_count) 
    # y RandomUnderSampler (undersample majority a target_majority_count)
    over = SMOTE(sampling_strategy={1: target_minority_count}, random_state=42)
    under = RandomUnderSampler(sampling_strategy={0: target_majority_count}, random_state=42)
    
    steps = [('o', over), ('u', under)]
    pipeline = Pipeline(steps=steps)
    
    X_res, y_res = pipeline.fit_resample(X, y)
    save_dataset(X_res, y_res, 'smote_undersampling', pct)

Generando SMOTE + Undersampling...
✅ Guardado: data\smote_undersampling\10.csv (Distribución Vishing: 10.00%) | Total: 47500 filas
✅ Guardado: data\smote_undersampling\20.csv (Distribución Vishing: 20.00%) | Total: 53437 filas
✅ Guardado: data\smote_undersampling\25.csv (Distribución Vishing: 25.00%) | Total: 57000 filas


## 8. CTGAN (Conditional Tabular GAN)

In [ ]:
print("Generando datos sintéticos con CTGAN...")

# Importamos joblib para forzar un backend sin multiprocesamiento para evitar errores de serialización
import joblib

# Solo entrenaremos la GAN con las sesiones de Vishing reales
df_vishing_only = pd.concat([X[y==1], y[y==1]], axis=1)

# Asegurarnos de que las columnas sean un Index 100% estándar (sin formatos especiales de pyarrow/pandas2)
df_vishing_only.columns = pd.Index([str(c) for c in df_vishing_only.columns], dtype=object)

# Creamos metadatos requeridos por SDV
metadata = SingleTableMetadata()
metadata.detect_from_dataframe(df_vishing_only)

# Entrenamiento GAN (Limitamos epochs para demostración - aumentar en producción a 300+)
synthesizer = CTGANSynthesizer(metadata, epochs=30, verbose=True)
print("\n--- Entrenando Modelo CTGAN ---")

# Envolver el entrenamiento en hilos en vez de procesos para que no intente usar un ProcessPool problemático
with joblib.parallel_backend('threading', n_jobs=1):
    synthesizer.fit(df_vishing_only)

for pct in TARGET_PERCENTAGES:
    # ¿Cuántas muestras extras de vishing necesitamos generar?
    target_minority_count = int(majority_count * (pct / (1 - pct)))
    n_to_generate = target_minority_count - minority_count
    
    if n_to_generate > 0:
        print(f"\nGenerando {n_to_generate} muestras sintéticas CTGAN para pct={pct*100}%...")
        synthetic_vishing = synthesizer.sample(num_rows=n_to_generate)
        
        # Purgamos también la salida generada
        synthetic_vishing.columns = [str(c) for c in synthetic_vishing.columns]
        
        # Retenemos el dataframe completo original y le anexamos las generadas
        df_final = pd.concat([df_original[features + [target_col]], synthetic_vishing], ignore_index=True)
        
        X_res = df_final[features]
        y_res = df_final[target_col]
        
        save_dataset(X_res, y_res, 'ctgan', pct)
    else:
        print(f"No se necesitan muestras sintéticas para pct={pct*100}%")

Generando datos sintéticos con CTGAN...

--- Entrenando Modelo CTGAN ---
